In [2]:
%pip install numpy
%pip install pandas

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: numpy in c:\users\silas\appdata\local\programs\python\python310\lib\site-packages (2.2.6)




[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.layers import LeakyReLU
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from tensorflow.keras.callbacks import ModelCheckpoint
import pandas as pd
import numpy as np
from lets_plot import *
LetsPlot.setup_html()

RuntimeError: module compiled against ABI version 0x1000009 but this version of numpy is 0x2000000

In [4]:
# Load in dataset
bikes = pd.read_csv('https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/bikes.csv')

In [5]:
# Load in crime datasets
import os

crime_stats = {}

path = "./crimestats/"
for f in os.listdir(path):
    if os.path.isfile(os.path.join(path, f)):
        year = f.replace('.csv', '')
        crime_stats[year] = pd.read_csv(os.path.join(path, f))
        


In [6]:
gas = pd.read_csv('data/gas_prices.csv')
gas['observation_date'] = pd.to_datetime(gas['observation_date'])
gas['year'] = gas['observation_date'].dt.year
gas['month'] = gas['observation_date'].dt.month

In [7]:
holidays_df = pd.read_csv('data/holidays.csv')
holidays_df['date'] = pd.to_datetime(holidays_df['date'])
holidays_df = holidays_df.drop_duplicates(subset='date')

In [8]:
bikes.head()

,dteday,hr,casual,registered,temp_c,feels_like_c,hum,windspeed,weathersit,season,holiday,workingday
0,1/1/2011,0.0,3,13,3.0,3.0,0.7957,0.8,1,1,0,0
1,1/1/2011,1.0,8,30,1.7,1.7,0.8272,0.8,1,1,0,0
2,1/1/2011,2.0,5,26,1.9,1.9,0.8157,1.1,1,1,0,0
3,1/1/2011,3.0,3,9,2.5,2.5,0.7831,0.8,1,1,0,0
4,1/1/2011,4.0,0,1,2.0,2.0,0.8075,1.1,1,1,0,0


In [9]:
# Feature Engineering ideas:
# Fix holidays
# Baseball Games for Nationals (home)

def process_df(df_to_process):
    df = df_to_process.copy()

    # Convert date to date
    df['dteday'] = pd.to_datetime(df['dteday'])

    # Pandemic years
    df['is_pandemic'] = ((df['dteday'] >= '3/30/2020') & (df['dteday'] <= '6/11/2021')).astype(int)

    # Cyclical Values
    # Hour
    df['hr_sin'] = np.sin(2 * np.pi * df['hr'] / 24)
    df['hr_cos'] = np.cos(2 * np.pi * df['hr'] / 24)

    # Season
    df['season_sin'] = np.sin(2 * np.pi * df['season'] / 4)
    df['season_cos'] = np.cos(2 * np.pi * df['season'] / 4)

    # Day of Week
    df['dayofweek'] = df['dteday'].dt.dayofweek
    df['dow_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)

    # Gas prices
    df['year'] = df['dteday'].dt.year
    df['month'] = df['dteday'].dt.month

    df = df.merge(gas[['year', 'month', 'cost_per_gallon']], on=['year', 'month'], how='left')

    # Crime data
    crime_all = pd.concat(crime_stats.values(), ignore_index=True)
    crime_all = crime_all.groupby('START_DATE', as_index=False)['count'].sum()
    crime_all['START_DATE'] = pd.to_datetime(crime_all['START_DATE'])

    df = df.merge(crime_all, left_on='dteday', right_on='START_DATE', how='left')
    df['count'] = df['count'].fillna(0)
    df = df.drop(columns=['START_DATE'])
    df = df.rename(columns={'count': 'crime_count'})

    # Other random crap
    df['temp_hr'] = df['temp_c'] * df['hr_cos']
    df['is_weekend'] = df['dayofweek'].isin([5,6]).astype(int)
    df['temp_workingday'] = df['temp_c'] * df['workingday']
    df['bad_weather'] = (df['weathersit'] >= 3).astype(int)
    df['temp_hum'] = df['temp_c'] * df['hum']
    df['is_peak_hour'] = df['hr'].isin([7,8,17,18]).astype(int)

    # Don't make these linear (extreme temps aren't presnet in linear scales)
    df['temp_sq'] = df['temp_c'] ** 2
    df['weather_season'] = df['weathersit'] * df['season']
    df['comfort_index'] = df['temp_c'] - (df['hum'] * 0.1) - (df['windspeed'] * 0.5)

    # Holidays
    '''
    df = df.merge(holidays_df, left_on='dteday', right_on='date', how='left')
    df = df.drop(columns=['date'])

    df['holiday_name'] = df['holiday_name'].fillna('none')
    df['holiday_name'] = (df['holiday_name']
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip('_'))

    dummies = pd.get_dummies(df['holiday_name'], prefix='is').astype(int)
    df = pd.concat([df.drop(columns=['holiday_name']), dummies], axis=1)
    '''

    # Drop columns
    df = df.drop(columns=['dteday', 'hr', 'season', 'year', 'month', 'hum', 'windspeed'])

    return df

clean_df = process_df(bikes)
clean_df.head(20)

,casual,registered,temp_c,feels_like_c,weathersit,holiday,workingday,is_pandemic,hr_sin,hr_cos,...,crime_count,temp_hr,is_weekend,temp_workingday,bad_weather,temp_hum,is_peak_hour,temp_sq,weather_season,comfort_index
0,3,13,3.0,3.0,1,0,0,0,0.000000e+00,1.000000e+00,...,96,3.000000e+00,1,0.0,0,2.38710,0,9.00,1,2.52043
1,8,30,1.7,1.7,1,0,0,0,2.588190e-01,9.659258e-01,...,96,1.642074e+00,1,0.0,0,1.40624,0,2.89,1,1.21728
2,5,26,1.9,1.9,1,0,0,0,5.000000e-01,8.660254e-01,...,96,1.645448e+00,1,0.0,0,1.54983,0,3.61,1,1.26843
3,3,9,2.5,2.5,1,0,0,0,7.071068e-01,7.071068e-01,...,96,1.767767e+00,1,0.0,0,1.95775,0,6.25,1,2.02169
4,0,1,2.0,2.0,1,0,0,0,8.660254e-01,5.000000e-01,...,96,1.000000e+00,1,0.0,0,1.61500,0,4.00,1,1.36925
5,0,1,2.7,1.3,1,0,0,0,9.659258e-01,2.588190e-01,...,96,6.988114e-01,1,0.0,0,2.07630,0,7.29,1,-0.07690
6,2,0,2.0,2.0,1,0,0,0,1.000000e+00,6.123234e-17,...,96,1.224647e-16,1,0.0,0,1.62620,0,4.00,1,1.41869
7,1,2,1.2,1.2,1,0,0,0,9.659258e-01,-2.588190e-01,...,96,-3.105829e-01,1,0.0,0,1.03200,1,1.44,1,0.61400
8,1,7,2.9,2.9,1,0,0,0,8.660254e-01,-5.000000e-01,...,96,-1.450000e+00,1,0.0,0,2.27766,1,8.41,1,2.12146
9,8,6,7.0,7.0,1,0,0,0,7.071068e-01,-7.071068e-01,...,96,-4.949747e+00,1,0.0,0,5.49290,0,49.00,1,5.82153


In [10]:
X = clean_df.drop(columns=['casual', 'registered'])
y = bikes[['casual', 'registered']]

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# scale X
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# scale Y
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train)
y_test_scaled = y_scaler.transform(y_test)

In [12]:
model = Sequential()
model.add(Dense(64, input_dim=X_train.shape[1], activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(16, activation='relu'))
model.add(Dense(2, activation='linear'))

c:\Users\Silas\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [13]:
# opt = keras.optimizers.Adam(learning_rate=0.0001)
opt = keras.optimizers.Adam(learning_rate=0.001)
model.compile(loss="mse", optimizer=opt, metrics=['mse'])

In [14]:
# increase batch size when exploring
checkpoint_callback = ModelCheckpoint(
    filepath="models/best_short_train.keras",
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_mse', # val_mse
    patience=30,
    restore_best_weights=True,
    verbose=1,
    mode='min'
)

# reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor='val_mse', factor=0.5, patience=15, min_lr=1e-5)

BATCH_SIZE = 32
EPOCHS = 500

history = model.fit(X_train_scaled, y_train_scaled, epochs=EPOCHS, validation_split=0.20, batch_size=BATCH_SIZE, callbacks=[early_stop, checkpoint_callback], shuffle=True)

Epoch 1/500
1931/1969 ━━━━━━━━━━━━━━━━━━━━ 0s 888us/step - loss: 0.4901 - mse: 0.4901
Epoch 1: val_loss improved from None to 0.23604, saving model to models/best_short_train.keras
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - loss: 0.3705 - mse: 0.3705 - val_loss: 0.2360 - val_mse: 0.2360
Epoch 2/500
1947/1969 ━━━━━━━━━━━━━━━━━━━━ 0s 883us/step - loss: 0.2793 - mse: 0.2793
Epoch 2: val_loss improved from 0.23604 to 0.22579, saving model to models/best_short_train.keras
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.2697 - mse: 0.2697 - val_loss: 0.2258 - val_mse: 0.2258
Epoch 3/500
1915/1969 ━━━━━━━━━━━━━━━━━━━━ 0s 873us/step - loss: 0.2480 - mse: 0.2480
Epoch 3: val_loss improved from 0.22579 to 0.20268, saving model to models/best_short_train.keras
1969/1969 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.2466 - mse: 0.2466 - val_loss: 0.2027 - val_mse: 0.2027
Epoch 4/500
1934/1969 ━━━━━━━━━━━━━━━━━━━━ 0s 890us/step - loss: 0.2359 - mse: 0.2359
Epoch 4: val_loss improved from 0.2026

In [15]:
# Create and train a bunch of models or smth idk
'''
configs = [
    {'layers': [64, 32, 16], 'dropout': 0.2, 'lr': 0.001},
    {'layers': [128, 64, 32], 'dropout': 0.3, 'lr': 0.001},
    {'layers': [32, 16], 'dropout': 0.1, 'lr': 0.0005},
    {'layers': [128, 64, 32, 16], 'dropout': 0.3, 'lr': 0.0005},
    {'layers': [64, 32, 16], 'dropout': 0.3, 'lr': 0.0001},
    {'layers': [96, 48, 24], 'dropout': 0.25, 'lr': 0.001},
]

for i, cfg in enumerate(configs):
    model = Sequential()
    model.add(Dense(cfg['layers'][0], input_dim=X_train_scaled.shape[1], activation='relu'))
    model.add(Dropout(cfg['dropout']))
    for units in cfg['layers'][1:]:
        model.add(Dense(units, activation='relu'))
        model.add(Dropout(cfg['dropout']))
    model.add(Dense(2, activation='linear'))

    opt = keras.optimizers.Adam(learning_rate=cfg['lr'])
    model.compile(loss="mean_squared_error", optimizer=opt, metrics=['mse'])

    early_stop = keras.callbacks.EarlyStopping(monitor='val_mse', patience=30, restore_best_weights=True)

    model.fit(X_train_scaled, y_train_scaled, epochs=500, validation_split=0.20,
              batch_size=32, callbacks=[early_stop], shuffle=True, verbose=0)

    model.save(f'models/model_{i}.keras')
    print(f"Model {i} done")
'''

'\nconfigs = [\n    {\'layers\': [64, 32, 16], \'dropout\': 0.2, \'lr\': 0.001},\n    {\'layers\': [128, 64, 32], \'dropout\': 0.3, \'lr\': 0.001},\n    {\'layers\': [32, 16], \'dropout\': 0.1, \'lr\': 0.0005},\n    {\'layers\': [128, 64, 32, 16], \'dropout\': 0.3, \'lr\': 0.0005},\n    {\'layers\': [64, 32, 16], \'dropout\': 0.3, \'lr\': 0.0001},\n    {\'layers\': [96, 48, 24], \'dropout\': 0.25, \'lr\': 0.001},\n]\n\nfor i, cfg in enumerate(configs):\n    model = Sequential()\n    model.add(Dense(cfg[\'layers\'][0], input_dim=X_train_scaled.shape[1], activation=\'relu\'))\n    model.add(Dropout(cfg[\'dropout\']))\n    for units in cfg[\'layers\'][1:]:\n        model.add(Dense(units, activation=\'relu\'))\n        model.add(Dropout(cfg[\'dropout\']))\n    model.add(Dense(2, activation=\'linear\'))\n\n    opt = keras.optimizers.Adam(learning_rate=cfg[\'lr\'])\n    model.compile(loss="mean_squared_error", optimizer=opt, metrics=[\'mse\'])\n\n    early_stop = keras.callbacks.EarlyStoppin

In [16]:
mse_df = pd.DataFrame({
    'epoch': range(len(history.history['mse'])),
    'Train': history.history['mse'],
    'Val': history.history['val_mse']
}).melt(id_vars='epoch', var_name='split', value_name='mse')

(ggplot(mse_df, aes('epoch', 'mse', color='split'))
 + geom_line()
 + ggtitle('Model MSE over Epochs')
 + xlab('Epoch')
 + ylab('MSE')
 + theme_minimal()
)

In [17]:
baseline_mse = ((model.predict(X_test_scaled) - y_test_scaled) ** 2).mean()

importances = {}
for col in X_test.columns:
    X_perm = X_test.copy()
    X_perm[col] = np.random.permutation(X_perm[col].values)
    X_perm_scaled = scaler.transform(X_perm)
    perm_mse = ((model.predict(X_perm_scaled) - y_test_scaled) ** 2).mean()
    importances[col] = perm_mse - baseline_mse

1055/1055 ━━━━━━━━━━━━━━━━━━━━ 1s 511us/step
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 1s 485us/step
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 1s 509us/step
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 1s 493us/step
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 1s 501us/step
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 1s 497us/step
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 1s 488us/step
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 1s 510us/step
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 1s 505us/step
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 1s 499us/step
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 1s 509us/step
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 1s 521us/step
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 1s 488us/step
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 1s 501us/step
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 1s 536us/step
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 1s 526us/step
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 1s 514us/step
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 1s 535us/step
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 1s 527us/step
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 1s 513us/step
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 1s 513us/step
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 1s 515us/step
1055/1055 

In [18]:
imp_df = pd.DataFrame(importances.items(), columns=['feature', 'importance']).sort_values('importance', ascending=False)

(ggplot(imp_df, aes(x='feature', y='importance'))
 + geom_bar(stat='identity', fill='#1D6FA4')
 + ggtitle('Feature Importance via Permutation')
 + xlab('Feature')
 + ylab('Increase in MSE')
 + theme_minimal()
 + theme(axis_text_x=element_text(angle=45, hjust=1)) +
 ggsize(1000, 600)
)

In [19]:
train_predictions = y_scaler.inverse_transform(model.predict(X_train_scaled))
train_pred_sum = train_predictions.sum(axis=1)
train_actual_sum = y_train.sum(axis=1)

bias_corrector = LinearRegression()
bias_corrector.fit(train_pred_sum.reshape(-1, 1), train_actual_sum)

2461/2461 ━━━━━━━━━━━━━━━━━━━━ 1s 512us/step


,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [20]:
holdout = pd.read_csv('https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/biking_holdout_test_mini.csv')
holdout_clean = process_df(holdout)

holdout_scaled = scaler.transform(holdout_clean)
predictions = y_scaler.inverse_transform(model.predict(holdout_scaled))
pred_sum = predictions.sum(axis=1)
pred_corrected = bias_corrector.predict(pred_sum.reshape(-1, 1))

results = pd.DataFrame({'total': pred_corrected})
results.to_csv('holdout/ctrl-alt-elite-module4-predictions.csv', index=False)

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


In [21]:
# Load in all the models in the models directory and make predictions and store them
holdout = pd.read_csv('https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/biking_holdout_test_mini.csv')
holdout_clean = process_df(holdout)
holdout_scaled = scaler.transform(holdout_clean)

train_actual_sum = y_train.sum(axis=1)

model_files = [f for f in os.listdir('models') if f.endswith('.keras')]

for f in model_files:
    m = keras.models.load_model(os.path.join('models', f))

    # Refit bias corrector for THIS model
    train_pred = y_scaler.inverse_transform(m.predict(X_train_scaled))
    train_pred_sum = train_pred.sum(axis=1)

    corrector = LinearRegression()
    corrector.fit(train_pred_sum.reshape(-1, 1), train_actual_sum)

    # Holdout predictions
    predictions = y_scaler.inverse_transform(m.predict(holdout_scaled))
    pred_sum = predictions.sum(axis=1)
    pred_corrected = corrector.predict(pred_sum.reshape(-1, 1))

    file_name = f.replace('.keras', '')
    results = pd.DataFrame({'total': pred_corrected})
    results.to_csv(f'holdout/{file_name}-predictions.csv', index=False)

2461/2461 ━━━━━━━━━━━━━━━━━━━━ 1s 518us/step
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 


ValueError: Exception encountered when calling Sequential.call().

[1mInput 0 of layer "dense_32" is incompatible with the layer: expected axis -1 of input shape to have value 23, but received input with shape (32, 24)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(32, 24), dtype=float32)
  • training=False
  • mask=None
  • kwargs=<class 'inspect._empty'>